# MRI 切片融合與 3D CNN

課程提交版本的公開整理副本。所有歷史輸出、附件及非必要 metadata 已移除。
先閱讀上層 README 的資料需求、已知問題與重現限制；不要直接 Run All。


# Homework 3 Problem 2: 核磁共振造影的 3D 影像分析


In [ ]:
# Notebook setup disabled in public copy: !pip install pydicom
# Notebook setup disabled in public copy: !pip install torchsummary

In [ ]:
import warnings
import gc
import os
import random

from tqdm import tqdm
import numpy as np
import pandas as pd
import cv2
import pydicom
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, Subset, DataLoader
from torchvision import models, transforms
from torchvision.transforms import v2
from torchsummary import summary

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

warnings.filterwarnings('ignore')

In [ ]:
def seed_everything(seed):
    # Set Python random seed
    random.seed(seed)

    # Set NumPy random seed
    np.random.seed(seed)

    # Set PyTorch random seed for CPU and GPU
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Set PyTorch deterministic operations for cudnn backend
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

## Data


In [ ]:
root = "/kaggle/input/mia-hw3"

train_data = pd.read_csv(os.path.join(root, "train.csv"))
print(f"Numbers of train data : {len(train_data)}")
train_data.head(10)

In [ ]:
train_data_label_dist = pd.DataFrame(
    {
        "label" : train_data.iloc[:, 1].value_counts().sort_index().index.astype(str),
        "counts" : train_data.iloc[:, 1].value_counts().sort_index().values
    }
)
train_data_label_dist

## Defined Image Interpolation


In [ ]:
# image interpolation
class Interpolate():

    def __init__(self, target_num = 22):
        self.target_num = target_num

    def __call__(self, slices): # (3, XX, 512, 512) 其中 XX 是切片的數量
        slices = slices.unsqueeze(dim = 0) # (1, 3, XX, 512, 512)
        slices = F.interpolate(
                  slices,
                  mode = 'trilinear',
                  size = (self.target_num, slices.shape[-2], slices.shape[-1])) # (1, 3, target_num, 512, 512) 其中 slices.shape[-2], slices.shape[-1]為 XX 張 slice 的 H, W
        return torch.transpose(slices.squeeze(), 0, 1) # slices.squeeze()的形狀為 (3, 22, 512, 512)， 經過 transpose 後形狀變為 (22, 3, 512, 512)

## Data Transformation


In [ ]:
def build_transform(is_train):
    t = []
    if is_train:
        t.append(Interpolate(target_num = 22))
        return v2.Compose(t)

    t.append(Interpolate(target_num = 22))
    return v2.Compose(t)

## Evaluator


In [ ]:
def evaluator(preds, gts):
    preds = preds.cpu().detach().numpy()
    gts = gts.cpu().numpy()
    acc = accuracy_score(preds, gts)
    f1 = f1_score(preds, gts, average="macro")

    return acc, f1

## Training


In [ ]:
def train_one_epoch(model, train_loader, optimizer, scheduler, criterion, device):
    model.train()
    train_loss = .0
    predictions, ground_truths = [], []
    for images, labels in train_loader:
        images = [image.to(device=device, dtype=torch.float) for image in images]
        labels = labels.to(device=device, dtype=torch.long)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()

        optimizer.step()
        # scheduler.step()

        train_loss += loss.item()
        preds = torch.argmax(logits, dim=1)

        predictions.append(preds)
        ground_truths.append(labels)

    scheduler.step()

    train_loss /= len(train_loader)

    predictions = torch.cat(predictions)
    ground_truths = torch.cat(ground_truths)
    train_acc, train_f1 = evaluator(predictions, ground_truths)

    return train_loss, 100*train_acc, 100*train_f1

## Validation


In [ ]:
def validation(model, valid_loader, criterion, device):
    model.eval()
    valid_loss = .0
    predictions, ground_truths = [], []
    with torch.no_grad():
        for images, labels in valid_loader:
            images = [image.to(device=device, dtype=torch.float) for image in images]
            labels = labels.to(device=device, dtype=torch.long)

            logits = model(images)
            loss = criterion(logits, labels)

            valid_loss += loss.item()
            preds = torch.argmax(logits, dim=1)

            predictions.append(preds)
            ground_truths.append(labels)

        valid_loss /= len(valid_loader)

        predictions = torch.cat(predictions)
        ground_truths = torch.cat(ground_truths)
        valid_acc, valid_f1 = evaluator(predictions, ground_truths)
    return valid_loss, 100*valid_acc, 100*valid_f1

## Testing


In [ ]:
def testing(model, test_loader, device):
    model.eval()
    preds = []
    with torch.no_grad():
        for images in test_loader:

            images = [image.to(device=device, dtype=torch.float) for image in images]
            logits = model(images)
            pred = torch.argmax(logits, dim=1)

            preds.append(pred)


    preds = torch.cat(preds)

    return preds

# 2D CNN


In [ ]:
class MRI2DDataset(Dataset):
    def __init__(self, df, mode = None, transforms = None):

        root = "/kaggle/input/mia-hw3"
        self.ids = np.array(df["ID"])
        self.labels = torch.from_numpy(np.array(df["Disease"]))
        self.mode = mode
        self.transforms = transforms

        paths = []
        for ID in np.array(df["ID"]):
            path = os.path.join(root, "DICOM", str(ID).split('.')[0].zfill(7))
            paths.append(path)

        self.images = []

        for path in tqdm(paths):
            all_slices = []
            T1_root = os.path.join(root, "DICOM", path, "T1")
            for filename in sorted(os.listdir(T1_root), key = lambda s: int(pydicom.dcmread(os.path.join(T1_root, s)).InstanceNumber)):
                T1_image = []
                T2_image = []

                # T1 images
                img = pydicom.dcmread(os.path.join(T1_root, filename))
                ww = img.WindowWidth
                wc = img.WindowCenter
                highest_visible_value = (ww + 2 * wc) / 2
                lowest_visible_value = highest_visible_value - ww

                img = img.pixel_array
                img = np.clip(img, lowest_visible_value, highest_visible_value)
                img = 255 * (img - lowest_visible_value) / ww
                img = cv2.resize(img, (256, 256))

                T1_image.append(img)

                # T2 images
                img = pydicom.dcmread(os.path.join(root, "DICOM", path, "T2", filename))
                ww = img.WindowWidth
                wc = img.WindowCenter
                highest_visible_value = (ww + 2 * wc) / 2
                lowest_visible_value = highest_visible_value - ww

                img = img.pixel_array
                img = np.clip(img, lowest_visible_value, highest_visible_value)
                img = 255 * (img - lowest_visible_value) / ww
                img = cv2.resize(img, (256, 256))

                T2_image.append(img)

                # T1T2 images
                T1T2_image = (T1_image[0] + T2_image[0]) / 2
                T1T2_image = torch.tensor(T1T2_image)

                T1_image = torch.tensor(T1_image[0])
                T2_image = torch.tensor(T2_image[0])

                all_slices.append(torch.stack((T1_image, T2_image, T1T2_image), dim = 0))

            slices = torch.stack(all_slices, dim = 1)

            if self.transforms:
                slices = self.transforms(slices)

            # 這邊有修改
            image = []
            for i in range(len(slices)):
                image.append(slices[i].squeeze().float())
            self.images.append(image)


    def __getitem__(self, index):

        if self.mode == "train":

            images = self.images[index]
            label = self.labels[index]

            return images, label

        elif self.mode == "valid":

            images = self.images[index]
            label = self.labels[index]

            return images, label

        elif self.mode == "test":

            images = self.images[index]

            return images

    def __len__(self):
        return(len(self.ids))

In [ ]:
root = "/kaggle/input/mia-hw3"
df = pd.read_csv(os.path.join(root,"train.csv"))
dataset_2D = MRI2DDataset(df = df, mode = "train",transforms = build_transform(True))

print(len(dataset_2D))
print(len(dataset_2D[0])) # dataset_2D[0]是一個tuple: ([(3, 256, 256), ...,(3, 256, 256)], label)
print(len(dataset_2D[0][0])) # dataset_2D[0][0] 則是 [(3, 256, 256), ...,(3, 256, 256)]
print(dataset_2D[0][0][0].shape) # dataset_2D[0][0][0] 則是 (3, 256, 256)

# 使用 del 刪除不需要的變數，以釋放出這些變數佔用的 CPU 記憶體。當變數被 del 刪除後，Python 會標記這些記憶體為可回收狀態。
del dataset_2D
# 強制回收內存
gc.collect()

## Late Fusion


In [ ]:
class config:
    
    root = "/kaggle/input/mia-hw3"
    valid_prob = 0.2
    batch_size = 4
    lr = 1e-4
    epochs = 20
    weight_decay = 1e-3
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    seed = 42
    
print(config.device)

### Late fusion model


In [ ]:
# 下方的 Late fusion 是基於這個 VGG16 的 pretrained model 進行修改，Early fusion 和 Single Slice也可用此修改。
vgg16 = models.vgg16(weights='IMAGENET1K_V1', progress = True)
print(vgg16)
del vgg16

In [ ]:
class Late_fusion(nn.Module):
    def __init__(self, num_classes, input_size = (3, 256, 256), features_grad = False): 
        super().__init__()

        # 取出vgg16中的特徵層
        vgg16 = models.vgg16(weights='IMAGENET1K_V1', progress = True)
        
        # 將 VGG16 的 classifier 層替換為 nn.Identity()，丟棄預設的分類層。
        vgg16.classifier = nn.Identity()
        
        # 可調整 feature map 的大小，注意到會影響分類層        
        vgg16.avgpool = nn.AdaptiveAvgPool2d(output_size=(1,1))
        
        # 固定/不固定特徵層的參數值（features_grad = False / features_grad = True）
        for param in vgg16.features.parameters():
            param.requires_grad = features_grad
            
        self.backend = vgg16
        
        # 增加分類層
        self.classifier = nn.Sequential(
          nn.Linear(512 * 1 * 1 * 22, num_classes) 
        )

    def forward(self, x): # x: 22 個 (batch_size, 3, 256, 256) 的 list
        
        feature_map_list = []
        for i in range(len(x)):
            feature_map_list.append(self.backend(x[i]))
        feature_map_concentrate = torch.cat(feature_map_list, dim=1)  # shape: (batch_size, 512 * 22, 7, 7)
        feature_map_concentrate = feature_map_concentrate.view(feature_map_concentrate.size(0), -1)  # shape: (batch_size, 512 * 22 * 7 * 7)
        outputs = self.classifier(feature_map_concentrate)

        return outputs

### Parameters of Late fusion model


In [ ]:
model_late = Late_fusion(num_classes = 2)
total_params = sum(p.numel() for p in model_late.parameters())
print("總參數量:", total_params)

### Late fusion 模型測試


In [ ]:
# 隨機生成一個 batch size 為 4 、切片大小為（3, 256, 256）的 3D image
x = [torch.randn(4,3,256,256) for _ in range(22)] # x: [(4,3,256,256),...,(4,3,256,256)]
output = model_late(x)
print(output, output.shape)
# 如果輸出一個大小為 torch.Size([4,2]) 的張量，表示測試通過
del model_late
gc.collect()

### Late fusion main


In [ ]:
def main():

    seed_everything(config.seed)
    
    df = pd.read_csv(os.path.join(config.root, "train.csv"))
    
    # split training & validation dataset 
    n = df.shape[0]
    valid_size = int(n * config.valid_prob)
    train_ids , valid_ids = train_test_split(
     np.linspace(0, n - 1, n).astype("int"),
     test_size = valid_size,
     random_state = config.seed,
    )
    
    print("Initializing dataset...")  
    train_df = df.iloc[train_ids]
    valid_df = df.iloc[valid_ids]
    train_dataset = MRI2DDataset(train_df, mode = "train",transforms = build_transform(True))
    valid_dataset = MRI2DDataset(valid_df, mode = "valid",transforms = build_transform(False))

    
    # DataLoader
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
    valid_loader = torch.utils.data.DataLoader(valid_dataset, batch_size=config.batch_size, shuffle=False)
    
    
    # settings
    print("Initializing model...")
    num_classes = 2
    print('num_classes',num_classes)
    model_late = Late_fusion(num_classes = num_classes, features_grad = True)
    model_late.to(config.device)

    criterion = nn.CrossEntropyLoss().to(config.device)
    
    parameters = [p for p in model_late.parameters() if p.requires_grad] 

    # 選擇適當的 optimizer, scheduler
    optimizer = torch.optim.Adam(parameters, lr = config.lr, weight_decay = config.weight_decay)
    
    # 以 epoch 為單位來更新學習率
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=12, gamma=0.1)
    # 以 batch 為單位來更新學習率
    # scheduler = torch.optim.lr_scheduler.OneCycleLR(
    #     optimizer = optimizer,
    #     epochs = config.epochs,
    #     steps_per_epoch = train_loader.__len__(),
    #     max_lr = config.lr,
    #     anneal_strategy = 'cos'
    # )

    # recordings
    best_val_loss = float("inf")
    history = {
      "train": {
          "loss": [],
          "acc": [],
          "f1": []
      },
      "valid": {
          "loss": [],
          "acc": [],
          "f1": []
      },
    }

    print("Start training!")
    for epoch in range(config.epochs):
        train_loss, train_acc, train_f1 = train_one_epoch(model_late, train_loader, optimizer, scheduler, criterion, config.device)
        valid_loss, valid_acc, valid_f1 = validation(model_late, valid_loader, criterion, config.device)

        # Log the loss and validation result
        history["train"]["loss"].append(train_loss)
        history["train"]["acc"].append(train_acc)
        history["train"]["f1"].append(train_f1)
        history["valid"]["loss"].append(valid_loss)
        history["valid"]["acc"].append(valid_acc)
        history["valid"]["f1"].append(valid_f1)

        print(f'Epoch[{epoch+1}/{config.epochs}], Train Loss: {train_loss:.4f}, Train Accuracy: {train_acc:.2f}%, Train F1: {train_f1:.2f}% | Valid Loss: {valid_loss:.4f}, Valid Accuracy: {valid_acc:.2f}%, Valid F1: {valid_f1:.2f}% | LR: {optimizer.state_dict()["param_groups"][0]["lr"]:.6f}')

        if valid_loss < best_val_loss:
            save_file = {
                "model": model_late.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "epoch": epoch,
                "args": config
            }
            best_val_loss = valid_loss
            torch.save(save_file, "checkpoint_Late.pth")

        best_ckpt = torch.load("checkpoint_Late.pth", map_location=config.device)
        model_late.load_state_dict(best_ckpt["model"])
    
    valid_loss, valid_acc, valid_f1 = validation(model_late, valid_loader, criterion, config.device)
    print("*"*10, " Validation ", "*"*10)
    print(f"Valid Loss: {valid_loss:.4f}, Valid Accuracy: {valid_acc:.2f}%, Valid F1: {valid_f1:.2f}%")
    
    del train_loader
    del valid_loader
    del train_dataset
    del valid_dataset
    gc.collect()
    # 用於釋放 CUDA GPU 的顯存
    torch.cuda.empty_cache()
    
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(range(config.epochs), history["train"]["loss"], label='Training Loss')
    plt.plot(range(config.epochs), history["valid"]["loss"], label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.title('Training and Validation Loss Curves')
    plt.show()

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(range(config.epochs), history["train"]["acc"], label='Training Acc')
    plt.plot(range(config.epochs), history["valid"]["acc"], label='Validation Acc')
    plt.xlabel('Epoch')
    plt.ylabel('ACC')
    plt.legend()
    plt.title('Training and Validation Accuracy Curves')
    plt.show()

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(range(config.epochs), history["train"]["f1"], label='Training F1')
    plt.plot(range(config.epochs), history["valid"]["f1"], label='Validation F1')
    plt.xlabel('Epoch')
    plt.ylabel('F1')
    plt.legend()
    plt.title('Training and Validation F1 Score Curves')
    plt.show()

if __name__ == "__main__":
    main()

In [ ]:
seed_everything(config.seed)
test_df = pd.read_csv(os.path.join(config.root, "test.csv"))

test_dataset = MRI2DDataset(test_df, mode = "test",transforms = build_transform(False))
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False)
model_late = Late_fusion(num_classes = 2, features_grad = True)

model_late.to("cuda")

best_ckpt = torch.load(f"checkpoint_Late.pth")
model_late.load_state_dict(best_ckpt["model"])

preds = testing(model_late, test_loader, config.device)
del test_dataset
gc.collect()
torch.cuda.empty_cache()

In [ ]:
results_df = pd.read_csv(os.path.join(config.root, "submission.csv"))
results_df['Disease'] = preds.cpu().numpy()
results_df.to_csv("late.csv", index=False)

## Single Slice


In [ ]:
class config:
    
    root = "/kaggle/input/mia-hw3"
    valid_prob = 0.2
    batch_size = 4
    lr = 1e-4
    epochs = 20
    weight_decay = 1e-3
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    seed = 42
    
print(config.device)

### Single slice model


In [ ]:
# 建立完整的 Single slice model
class Single_slice(nn.Module):
    def __init__(self, num_classes, input_size = (3, 256, 256), features_grad = False):
        super().__init__()

        # 取出vgg16中的特徵層
        vgg16 = models.vgg16(weights='IMAGENET1K_V1', progress = True)
        vgg16.classifier = nn.Identity()
        vgg16.avgpool = nn.AdaptiveAvgPool2d(output_size=(1,1))

        # 固定/不固定特徵層的參數值
        for param in vgg16.features.parameters():
            param.requires_grad = features_grad
        self.backend = vgg16

        # 修改分類層
        self.classifier = nn.Linear(512 * 1 * 1, num_classes)
        self.softmax = nn.Softmax(dim=1)

    # 完成 foward 函數
    def forward(self, x): # x: 22 個 (batch_size, 3, 256, 256) 的 list
        feature_maps = [self.backend(img) for img in x]
        softmax_results = [self.softmax(self.classifier(fm)) for fm in feature_maps]
        output = torch.mean(torch.stack(softmax_results), dim=0)
        log_output = torch.log(output + 1e-10)
        return log_output

### Parameters of Single slice model


In [ ]:
model = Single_slice(num_classes = 2)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("總參數量:", total_params)

### Single slice main


In [ ]:
# nn.CrossEntropyLoss() 改成 nn.NLLLoss()
def main():

    seed_everything(config.seed)

    df = pd.read_csv(os.path.join(config.root, "train.csv"))

    # split training & validation dataset
    n = df.shape[0]
    valid_size = int(n * config.valid_prob)
    train_ids , valid_ids = train_test_split(
     np.linspace(0, n - 1, n).astype("int"),
     test_size = valid_size,
     random_state = config.seed,
    )

    print("Initializing dataset...")
    train_df = df.iloc[train_ids]
    valid_df = df.iloc[valid_ids]
    train_dataset = MRI2DDataset(train_df, mode = "train",transforms = build_transform(True))
    valid_dataset = MRI2DDataset(valid_df, mode = "valid",transforms = build_transform(False))


    # DataLoader
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
    valid_loader = torch.utils.data.DataLoader(valid_dataset, batch_size=config.batch_size, shuffle=False)


    # settings
    print("Initializing model...")
    num_classes = 2
    print('num_classes',num_classes)
    model_single = Single_slice(num_classes = num_classes, features_grad = True)
    model_single.to(config.device)

    # 散則適當的criterion, optimizer
    criterion = nn.NLLLoss().to(config.device)
    optimizer = torch.optim.Adam(model_single.parameters(), lr = config.lr, weight_decay = config.weight_decay)
    # 以 epoch 為單位來更新學習率
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=12, gamma=0.1)
    # 以 batch 為單位來更新學習率
    # scheduler = torch.optim.lr_scheduler.OneCycleLR(
    #     optimizer = optimizer,
    #     epochs = config.epochs,
    #     steps_per_epoch = train_loader.__len__(),
    #     max_lr = config.lr,
    #     anneal_strategy = 'cos'
    # )

    # recordings
    best_val_loss = float("inf")
    history = {
      "train": {
          "loss": [],
          "acc": [],
          "f1": []
      },
      "valid": {
          "loss": [],
          "acc": [],
          "f1": []
      },
    }

    print("Start training!")
    for epoch in range(config.epochs):
        train_loss, train_acc, train_f1 = train_one_epoch(model_single, train_loader, optimizer, scheduler, criterion, config.device)
        valid_loss, valid_acc, valid_f1 = validation(model_single, valid_loader, criterion, config.device)

        # Log the loss and validation result
        history["train"]["loss"].append(train_loss)
        history["train"]["acc"].append(train_acc)
        history["train"]["f1"].append(train_f1)
        history["valid"]["loss"].append(valid_loss)
        history["valid"]["acc"].append(valid_acc)
        history["valid"]["f1"].append(valid_f1)

        print(f'Epoch[{epoch+1}/{config.epochs}], Train Loss: {train_loss:.4f}, Train Accuracy: {train_acc:.2f}%, Train F1: {train_f1:.2f}% | Valid Loss: {valid_loss:.4f}, Valid Accuracy: {valid_acc:.2f}%, Valid F1: {valid_f1:.2f}% | LR: {optimizer.state_dict()["param_groups"][0]["lr"]:.6f}')

        if valid_loss < best_val_loss:
            save_file = {
                "model": model_single.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "epoch": epoch,
                "args": config
            }
            best_val_loss = valid_loss
            torch.save(save_file, "checkpoint_Single.pth")

        best_ckpt = torch.load("checkpoint_Single.pth", map_location=config.device)
        model_single.load_state_dict(best_ckpt["model"])

    valid_loss, valid_acc, valid_f1 = validation(model_single, valid_loader, criterion, config.device)
    print("*"*10, " Validation ", "*"*10)
    print(f"Valid Loss: {valid_loss:.4f}, Valid Accuracy: {valid_acc:.2f}%, Valid F1: {valid_f1:.2f}%")

    del train_loader
    del valid_loader
    del train_dataset
    del valid_dataset
    gc.collect()
    torch.cuda.empty_cache()

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(range(config.epochs), history["train"]["loss"], label='Training Loss')
    plt.plot(range(config.epochs), history["valid"]["loss"], label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.title('Training and Validation Loss Curves')
    plt.show()

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(range(config.epochs), history["train"]["acc"], label='Training Acc')
    plt.plot(range(config.epochs), history["valid"]["acc"], label='Validation Acc')
    plt.xlabel('Epoch')
    plt.ylabel('ACC')
    plt.legend()
    plt.title('Training and Validation Accuracy Curves')
    plt.show()

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(range(config.epochs), history["train"]["f1"], label='Training F1')
    plt.plot(range(config.epochs), history["valid"]["f1"], label='Validation F1')
    plt.xlabel('Epoch')
    plt.ylabel('F1')
    plt.legend()
    plt.title('Training and Validation F1 Score Curves')
    plt.show()

if __name__ == "__main__":
    main()

In [ ]:
seed_everything(config.seed)
test_df = pd.read_csv(os.path.join(config.root, "test.csv"))

test_dataset = MRI2DDataset(test_df, mode = "test",transforms = build_transform(False))
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False)
model_single = Single_slice(num_classes = 2, features_grad = True)

model_single.to("cuda")

best_ckpt = torch.load(f"checkpoint_Single.pth")
model_single.load_state_dict(best_ckpt["model"])

preds = testing(model_single, test_loader, config.device)
del test_dataset
gc.collect()
torch.cuda.empty_cache()

In [ ]:
results_df = pd.read_csv(os.path.join(config.root, "submission.csv"))
results_df['Disease'] = preds.cpu().numpy()
results_df.to_csv("single.csv", index=False)

## Early Fusion


In [ ]:
class config:
    
    root = "/kaggle/input/mia-hw3"
    valid_prob = 0.2
    batch_size = 4
    lr = 1e-4
    epochs = 20
    weight_decay = 1e-3
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    seed = 42
    
print(config.device)

### Early fusion model


In [ ]:
class Early_fusion(nn.Module):
    def __init__(self, num_classes, input_size = (3, 256, 256), features_grad = False):
        super().__init__()

        # 取出vgg16中的特徵層
        vgg16 = models.vgg16(weights='IMAGENET1K_V1', progress = True)
        # 將 VGG16 的 classifier 層替換為 nn.Identity()，丟棄預設的分類層。
        vgg16.classifier = nn.Identity()

        # 可調整 feature map 的大小，可以改變參數量，但注意到會影響分類層
        vgg16.avgpool = nn.AdaptiveAvgPool2d(output_size=(1,1))

        # 固定/不固定特徵層的參數值
        # 若 features_grad 設置為 FALSE，使用預訓練的 VGG16 特徵參數值，表示在訓練時參數將不會被更新
        # 若 features_grad 設置為 True，將利用預訓練的權重作為起點，然後讓模型根據本次的資料集進行調整。
        for param in vgg16.features.parameters():
            param.requires_grad = features_grad

        self.backend = vgg16

        # vgg16 最後一層通過 avgpool 後，每張 2D影像產生一個 (512, 7, 7) 的 feature map，共 22 個
        # 另外我們會對 22 張 feature map 進行拼接再攤平，得到一個長度為 512 * 22 * 7 * 7 的特徵向量

        # 增加分類層
        self.classifier = nn.Sequential(
          nn.Linear(512 * 1 * 1 * 22, num_classes)
        )

    # 在 PyTorch 中，forward 函數在 nn.Module 類的子類中是用來定義模型前向傳播過程的函數。這個函數描述了當 input 輸入模型時，如何一步一步地經過各層，最後輸出 outputs。
    def forward(self, x): # x: 22 個 (batch_size, 3, 256, 256) 的 list

        feature_map_list = []
        for i in range(len(x)):
            feature_map_list.append(self.backend(x[i]))
        feature_map_concentrate = torch.cat(feature_map_list, dim=1)  # shape: (batch_size, 512 * 22, 7, 7)
        feature_map_concentrate = feature_map_concentrate.view(feature_map_concentrate.size(0), -1)  # shape: (batch_size, 512 * 22 * 7 * 7)
        outputs = self.classifier(feature_map_concentrate)

        return outputs

### Parameters of Early fusion model


In [ ]:
model = Early_fusion(num_classes = 2)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("總參數量:", total_params)

### Early fusion main


In [ ]:
def main():

    seed_everything(config.seed)

    df = pd.read_csv(os.path.join(config.root, "train.csv"))

    # split training & validation dataset
    n = df.shape[0]
    valid_size = int(n * config.valid_prob)
    train_ids , valid_ids = train_test_split(
     np.linspace(0, n - 1, n).astype("int"),
     test_size = valid_size,
     random_state = config.seed,
    )

    print("Initializing dataset...")
    train_df = df.iloc[train_ids]
    valid_df = df.iloc[valid_ids]
    train_dataset = MRI2DDataset(train_df, mode = "train",transforms = build_transform(True))
    valid_dataset = MRI2DDataset(valid_df, mode = "valid",transforms = build_transform(False))


    # DataLoader
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
    valid_loader = torch.utils.data.DataLoader(valid_dataset, batch_size=config.batch_size, shuffle=False)


    # settings
    print("Initializing model...")
    num_classes = 2
    print('num_classes',num_classes)
    model_early = Early_fusion(num_classes = num_classes, features_grad = True)
    model_early.to(config.device)

    # 散則適當的criterion, optimizer
    criterion = nn.CrossEntropyLoss().to(config.device)
    optimizer = torch.optim.Adam(model_early.parameters(), lr = config.lr, weight_decay = config.weight_decay)
    # 以 epoch 為單位來更新學習率
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=12, gamma=0.1)
    # 以 batch 為單位來更新學習率
    # scheduler = torch.optim.lr_scheduler.OneCycleLR(
    #     optimizer = optimizer,
    #     epochs = config.epochs,
    #     steps_per_epoch = train_loader.__len__(),
    #     max_lr = config.lr,
    #     anneal_strategy = 'cos'
    # )

    # recordings
    best_val_loss = float("inf")
    history = {
      "train": {
          "loss": [],
          "acc": [],
          "f1": []
      },
      "valid": {
          "loss": [],
          "acc": [],
          "f1": []
      },
    }

    print("Start training!")
    for epoch in range(config.epochs):
        train_loss, train_acc, train_f1 = train_one_epoch(model_early, train_loader, optimizer, scheduler, criterion, config.device)
        valid_loss, valid_acc, valid_f1 = validation(model_early, valid_loader, criterion, config.device)

        # Log the loss and validation result
        history["train"]["loss"].append(train_loss)
        history["train"]["acc"].append(train_acc)
        history["train"]["f1"].append(train_f1)
        history["valid"]["loss"].append(valid_loss)
        history["valid"]["acc"].append(valid_acc)
        history["valid"]["f1"].append(valid_f1)

        print(f'Epoch[{epoch+1}/{config.epochs}], Train Loss: {train_loss:.4f}, Train Accuracy: {train_acc:.2f}%, Train F1: {train_f1:.2f}% | Valid Loss: {valid_loss:.4f}, Valid Accuracy: {valid_acc:.2f}%, Valid F1: {valid_f1:.2f}% | LR: {optimizer.state_dict()["param_groups"][0]["lr"]:.6f}')

        if valid_loss < best_val_loss:
            save_file = {
                "model": model_early.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "epoch": epoch,
                "args": config
            }
            best_val_loss = valid_loss
            torch.save(save_file, "checkpoint_Early.pth")

        best_ckpt = torch.load("checkpoint_Early.pth", map_location=config.device)
        model_early.load_state_dict(best_ckpt["model"])

    valid_loss, valid_acc, valid_f1 = validation(model_early, valid_loader, criterion, config.device)
    print("*"*10, " Validation ", "*"*10)
    print(f"Valid Loss: {valid_loss:.4f}, Valid Accuracy: {valid_acc:.2f}%, Valid F1: {valid_f1:.2f}%")


    del train_loader
    del valid_loader
    del train_dataset
    del valid_dataset
    gc.collect()
    torch.cuda.empty_cache()

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(range(config.epochs), history["train"]["loss"], label='Training Loss')
    plt.plot(range(config.epochs), history["valid"]["loss"], label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.title('Training and Validation Loss Curves')
    plt.show()

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(range(config.epochs), history["train"]["acc"], label='Training Acc')
    plt.plot(range(config.epochs), history["valid"]["acc"], label='Validation Acc')
    plt.xlabel('Epoch')
    plt.ylabel('ACC')
    plt.legend()
    plt.title('Training and Validation Accuracy Curves')
    plt.show()

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(range(config.epochs), history["train"]["f1"], label='Training F1')
    plt.plot(range(config.epochs), history["valid"]["f1"], label='Validation F1')
    plt.xlabel('Epoch')
    plt.ylabel('F1')
    plt.legend()
    plt.title('Training and Validation F1 Score Curves')
    plt.show()

if __name__ == "__main__":
    main()

In [ ]:
seed_everything(config.seed)
test_df = pd.read_csv(os.path.join(config.root, "test.csv"))

test_dataset = MRI2DDataset(test_df, mode = "test",transforms = build_transform(False))
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False)
model_early = Late_fusion(num_classes = 2, features_grad = True)

model_early.to("cuda")

best_ckpt = torch.load(f"checkpoint_Early.pth")
model_early.load_state_dict(best_ckpt["model"])

preds = testing(model_early, test_loader, config.device)
del test_dataset
gc.collect()
torch.cuda.empty_cache()

In [ ]:
results_df = pd.read_csv(os.path.join(config.root, "submission.csv"))
results_df['Disease'] = preds.cpu().numpy()
results_df.to_csv("early.csv", index=False)

# 3D CNN


In [ ]:
class config:
    
    root = "/kaggle/input/mia-hw3"
    valid_prob = 0.2
    batch_size = 4
    lr = 1e-4
    epochs = 20
    weight_decay = 1e-3
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    seed = 42
    
print(config.device)

In [ ]:
class MRI3DDataset(torch.utils.data.Dataset):
    def __init__(self, df, mode = None, transforms = None):

        root = "/kaggle/input/mia-hw3"
        self.ids = np.array(df["ID"])
        self.labels = torch.tensor(df["Disease"].values)
        self.mode = mode
        self.transforms = transforms

        paths = []
        for ID in self.ids:
            path = os.path.join(root, "DICOM", str(ID).split('.')[0].zfill(7))
            paths.append(path)

        self.images = []

        for path in tqdm(paths):
            all_slices = []
            T1_root = os.path.join(root, "DICOM", path, "T1")
            T2_root = os.path.join(root, "DICOM", path, "T2")
            for filename in sorted(os.listdir(T1_root), key = lambda s: int(pydicom.dcmread(os.path.join(T1_root, s)).InstanceNumber)):
                T1_image = []
                T2_image = []

                # T1 images
                img = pydicom.dcmread(os.path.join(T1_root, filename))
                ww = img.WindowWidth
                wc = img.WindowCenter
                highest_visible_value = (ww + 2 * wc) / 2
                lowest_visible_value = highest_visible_value - ww

                img = img.pixel_array
                img = np.clip(img, lowest_visible_value, highest_visible_value)
                img = 255 * (img - lowest_visible_value) / ww
                img = cv2.resize(img, (256, 256))

                T1_image.append(img)

                # T2 images
                img = pydicom.dcmread(os.path.join(T2_root, filename))
                ww = img.WindowWidth
                wc = img.WindowCenter
                highest_visible_value = (ww + 2 * wc) / 2
                lowest_visible_value = highest_visible_value - ww

                img = img.pixel_array
                img = np.clip(img, lowest_visible_value, highest_visible_value)
                img = 255 * (img - lowest_visible_value) / ww
                img = cv2.resize(img, (256, 256))

                T2_image.append(img)

                # T1T2 images
                T1T2_image = (T1_image[0] + T2_image[0]) / 2
                T1T2_image = torch.tensor(T1T2_image)

                T1_image = torch.tensor(T1_image[0])
                T2_image = torch.tensor(T2_image[0])

                all_slices.append(torch.stack((T1_image, T2_image, T1T2_image), dim = 0))
                # 其中 torch.stack((T1_image, T2_image, T1T2_image), dim = 0) 的形狀是 (3, 512, 512)

            slices = torch.stack(all_slices, dim = 1) # (3, XX, 512, 512) XX為原始切片數量

            if self.transforms:
                slices = self.transforms(slices) # (22, 3, 256, 256)
                slices = torch.transpose(slices, 0, 1) # (3, 22, 256, 256)

            self.images.append(slices.float())

    def __getitem__(self, index):

        if self.mode == "train":

            images = self.images[index]
            label = self.labels[index]

            return images, label

        elif self.mode == "valid":

            images = self.images[index]
            label = self.labels[index]

            return images, label

        elif self.mode == "test":

            images = self.images[index]

            return images

    def __len__(self):
        return(len(self.ids))

In [ ]:
train_df = pd.read_csv(os.path.join(config.root,"train.csv"))
print("Initializing dataset...")
dataset = MRI3DDataset(train_df, mode = "train", transforms = build_transform(True))

print(dataset[0][0].shape)
del dataset
gc.collect()

## 3D CNN model


In [ ]:
# 載入github資源
# Notebook setup disabled in public copy: !git clone https://github.com/Tencent/MedicalNet.git
import sys
sys.path.append('/kaggle/working/MedicalNet')
from models import resnet

In [ ]:
# 層數可自行決定
def generate_model(model_type='resnet', model_depth=50,
                   input_W=224, input_H=224, input_D=224, shortcut_type='B',
                   no_cuda=True, gpu_id=[0],
                   pretrain_path = 'pretrain/resnet_50.pth',
                   nb_class=1):

    # shortcut_type = "B"在 ResNet 中指的是一種捷徑（shortcut）連接方法，用來讓輸入和輸出尺寸一致。

    # 因為我們是調用上面 github 提供的模型，因為只有提供 resnet 所以這邊確認 model_type 是 resnet
    assert model_type in [
        'resnet'
    ]
    # 當 model_type 確認為 resnet 時，再檢查 model_depth 是否在指定的範圍內（10、18、34、50、101、152、200）
    if model_type == 'resnet':
        assert model_depth in [10, 18, 34, 50, 101, 152, 200]

    model = resnet.resnet50(sample_input_W=input_W, sample_input_H=input_H, sample_input_D=input_D, num_seg_classes = nb_class)

    # 修改原本的 conv_seg 模組來應用到我們的分類任務
    model.conv_seg = nn.Sequential(nn.AdaptiveAvgPool3d((1, 1, 1)), nn.Flatten())
    # nn.AdaptiveAvgPool3d((1, 1, 1))：這層會將輸入的 feature map 進行 Global Average Pooling，將維度 (D, H, W) 縮小到 (1, 1, 1)。最終輸出會是形狀 (batchsize, channels, 1, 1, 1)
    # nn.Flatten()：將 pooling 後的輸出攤成一個二維向量，形狀是 (batchsize, channels)


    if not no_cuda:
        if len(gpu_id) > 1:
            model = model.cuda()
            model = nn.DataParallel(model, device_ids=gpu_id)
            net_dict = model.state_dict()
        else:
            import os
            os.environ["CUDA_VISIBLE_DEVICES"]=str(gpu_id[0])
            model = model.cuda()
            model = nn.DataParallel(model, device_ids=None)
            net_dict = model.state_dict()
    else:
        net_dict = model.state_dict()

    print('loading pretrained model {}'.format(pretrain_path))
    pretrain = torch.load(pretrain_path)
    pretrain_dict = {k: v for k, v in pretrain['state_dict'].items() if k in net_dict.keys()}

    net_dict.update(pretrain_dict)

    return model

In [ ]:
resnet50_3d = generate_model(model_type='resnet', model_depth=50,
                 input_W=256, input_H=256, input_D=22, shortcut_type='B',
                 pretrain_path = "/kaggle/input/medicalnet/Pretrained/Pretrained/resnet_50.pth",
                 nb_class=2)

resnet50_3d.classifier = nn.Identity()
print(resnet50_3d)

In [ ]:
class Resnet3d(nn.Module):
    def __init__(self, num_classes, features_grad = False):
        super().__init__()

         # 用上方定義 generate_model 函數生成一個預訓練的 3D ResNet50 模型。
        resnet50_3d = generate_model(model_type='resnet', model_depth=50,
                         input_W=256, input_H=256, input_D=22, shortcut_type='B',
                         pretrain_path = "/kaggle/input/medicalnet/Pretrained/Pretrained/resnet_50.pth",
                         nb_class=2)

        resnet50_3d.classifier = nn.Identity()

        for param in resnet50_3d.parameters():
            param.requires_grad = features_grad

        # 更改成 3 channel
        temp = [param for param in resnet50_3d.conv1.parameters()][0] # (64, 1, 7, 7, 7)
        temp = torch.cat((temp, temp, temp), dim = 1) # (64, 3, 7, 7, 7)
        resnet50_3d.conv1 = nn.Conv3d(3, 64, kernel_size=(7, 7, 7), stride=(2, 2, 2), padding=(3, 3, 3), bias=False)
        for param in resnet50_3d.conv1.parameters():
            param = temp

        self.backend = resnet50_3d
        self.classifier = nn.Sequential(
          nn.Linear(2048, num_classes)
        )

    def forward(self, x): # x 是一個 (batchsize, 3, 22, 256, 256) 的 tensor
        if type(x) == list:
            x = torch.stack(x, dim=0)
        output = self.backend(x)
        outputs = self.classifier(output)
        return outputs

In [ ]:
model_3d = Resnet3d(num_classes = 2)
total_params = sum(p.numel() for p in model_3d.parameters())
print("總參數量:", total_params)

In [ ]:
model_3d = Resnet3d(num_classes = 2)
model_3d.to("cuda")

# 隨機一個 (batchsize=4, 3, 22, 256, 256) 的 tensor
x = torch.randn((4, 3, 22, 256, 256)).to("cuda")
output = model_3d(x)
print(output)

del model_3d, x
gc.collect()
torch.cuda.empty_cache()

In [ ]:
model = Resnet3d(num_classes = 2)
model.to("cuda")
summary(model, input_size = (3, 22, 256, 256))

In [ ]:
def main():

    seed_everything(config.seed)

    df = pd.read_csv(os.path.join(config.root, "train.csv"))

    # split training & validation dataset
    n = df.shape[0]
    valid_size = int(n * config.valid_prob)
    train_ids , valid_ids = train_test_split(
     np.linspace(0, n - 1, n).astype("int"),
     test_size = valid_size,
     random_state = config.seed,
    )

    # DataLoader
    train_data = df.iloc[train_ids]
    valid_data = df.iloc[valid_ids]
    train_dataset = MRI3DDataset(train_data, mode = "train", transforms = build_transform(True))
    valid_dataset = MRI3DDataset(valid_data, mode = "valid", transforms = build_transform(False))

    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
    valid_loader = torch.utils.data.DataLoader(valid_dataset, batch_size=config.batch_size, shuffle=False)


    # settings
    print("Initializing model...")
    num_classes = 2
    print('num_classes',num_classes)
    model = Resnet3d(num_classes = num_classes, features_grad = False)
    model.to(config.device)
    criterion = nn.CrossEntropyLoss().to(config.device)

    # 選擇適當的 optimizer, scheduler
    optimizer = torch.optim.Adam(model.parameters(), lr = config.lr, weight_decay = config.weight_decay)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer = optimizer,
        epochs = config.epochs,
        steps_per_epoch = train_loader.__len__(),
        max_lr = config.lr,
        anneal_strategy = 'cos'
    )

    # recordings
    best_val_loss = float("inf")
    history = {
      "train": {
          "loss": [],
          "acc": [],
          "f1": []
      },
      "valid": {
          "loss": [],
          "acc": [],
          "f1": []
      },
    }

    for epoch in range(config.epochs):
        train_loss, train_acc, train_f1 = train_one_epoch(model, train_loader, optimizer, scheduler, criterion, config.device)
        valid_loss, valid_acc, valid_f1 = validation(model, valid_loader, criterion, config.device)

        # Log the loss and validation result
        history["train"]["loss"].append(train_loss)
        history["train"]["acc"].append(train_acc)
        history["train"]["f1"].append(train_f1)
        history["valid"]["loss"].append(valid_loss)
        history["valid"]["acc"].append(valid_acc)
        history["valid"]["f1"].append(valid_f1)

        print(f'Epoch[{epoch+1}/{config.epochs}], Train Loss: {train_loss:.4f}, Train Accuracy: {train_acc:.2f}%, Train F1: {train_f1:.2f}% | Valid Loss: {valid_loss:.4f}, Valid Accuracy: {valid_acc:.2f}%, Valid F1: {valid_f1:.2f}% | LR: {optimizer.state_dict()["param_groups"][0]["lr"]:.6f}')

        if valid_loss < best_val_loss:
            save_file = {
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "epoch": epoch,
                "args": config
            }
            best_val_loss = valid_loss
            torch.save(save_file, "checkpoint_3D.pth")

        best_ckpt = torch.load("checkpoint_3D.pth", map_location=config.device)
        model.load_state_dict(best_ckpt["model"])

    valid_loss, valid_acc, valid_f1 = validation(model, valid_loader, criterion, config.device)
    print("*"*10, " Validation ", "*"*10)
    print(f"Valid Loss: {valid_loss:.4f}, Valid Accuracy: {valid_acc:.2f}%, Valid F1: {valid_f1:.2f}%")

    del train_dataset
    del valid_dataset
    del train_loader
    del valid_loader
    gc.collect()
    torch.cuda.empty_cache()

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(range(config.epochs), history["train"]["loss"], label='Training Loss')
    plt.plot(range(config.epochs), history["valid"]["loss"], label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.title('Training and Validation Loss Curves')
    plt.show()

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(range(config.epochs), history["train"]["acc"], label='Training Acc')
    plt.plot(range(config.epochs), history["valid"]["acc"], label='Validation Acc')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.title('Training and Validation Accuracy Curves')
    plt.show()

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(range(config.epochs), history["train"]["f1"], label='Training F1')
    plt.plot(range(config.epochs), history["valid"]["f1"], label='Validation F1')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.title('Training and Validation F1 Score Curves')
    plt.show()

if __name__ == "__main__":
    main()

In [ ]:
test_df = pd.read_csv(os.path.join(config.root, "test.csv"))
test_dataset = MRI3DDataset(test_df, mode = "test", transforms = build_transform(False))
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False)

model = Resnet3d(num_classes = 2).to("cuda")

best_ckpt = torch.load("checkpoint_3D.pth", map_location=config.device)
model.load_state_dict(best_ckpt["model"])

preds = testing(model, test_loader, config.device)

In [ ]:
results_df = pd.read_csv(os.path.join(config.root, "submission.csv"))
results_df['Disease'] = preds.cpu().numpy()
results_df.to_csv("3D.csv", index=False)